# Crop yearbook — cleaning

Raw file: `Crop_data/crop-wise-area-production-yield.csv`.


1. loads the yearbook and drops rows with no district / crop / year / season
2. builds a simple name key (lowercase, no spaces/punctuation)
3. matches that key onto `data/INDIA_DISTRICTS.geojson`
4. collapses duplicate rows
5. writes `data/processed/crop_clean.csv`



In [1]:
from pathlib import Path
import json
import re

import pandas as pd

raw_path = Path("Crop_data/crop-wise-area-production-yield.csv")
geo_path = Path("data/INDIA_DISTRICTS.geojson")
out_path = Path("data/processed/crop_clean.csv")
unmatched_path = Path("data/processed/crop_unmatched_districts.csv")

out_path.parent.mkdir(parents=True, exist_ok=True)

## Load



In [2]:
df = pd.read_csv(raw_path, low_memory=False)
print(df.shape)
print(df.columns.tolist())
print("years", df["year"].nunique(), df["year"].min(), "..", df["year"].max())
print(df["season"].value_counts())
df.head(3)

(455359, 16)
['id', 'year', 'state_name', 'state_code', 'district_name', 'district_code', 'crop_name', 'crop_code', 'crop_type', 'season', 'area', 'area_unit', 'production', 'production_unit', 'yield', 'yield_unit']
years 26 1997-1998 .. 2022-2023
season
Kharif        161402
Rabi          120049
Whole Year     81403
Total          48044
Summer         26811
Winter          9616
Autumn          8034
Name: count, dtype: int64


,id,year,state_name,state_code,district_name,district_code,crop_name,crop_code,crop_type,season,area,area_unit,production,production_unit,yield,yield_unit
0,0,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Arhar/Tur,202.0,Pulses,Kharif,21400.0,Hectare,2600.0,Tonnes,0.121,Tonnes/Hectare
1,1,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Bajra,103.0,Cereals,Kharif,1400.0,Hectare,500.0,Tonnes,0.357,Tonnes/Hectare
2,2,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Castor Seed,1002.0,Oilseeds,Kharif,1000.0,Hectare,100.0,Tonnes,0.100,Tonnes/Hectare


## Drop empty keys

If district, crop, year or season is blank, that row cannot go on the map or into a yearly panel.

In [3]:
n0 = len(df)
df = df.copy()

for col in ["year", "state_name", "district_name", "crop_name", "season"]:
    df[col] = df[col].astype(str).str.strip()

df = df[
    df["district_name"].ne("") &
    df["district_name"].str.lower().ne("nan") &
    df["crop_name"].ne("") &
    df["year"].ne("") &
    df["season"].ne("")
]
print("dropped", n0 - len(df), "rows with empty keys")

dropped 0 rows with empty keys


##  Make a join key from a name

Crop file: `Ahmedabad`, `Bengaluru Rural`.
Map file: `AHMADABAD`, `Bengal#ru` (`#` is a broken `u` from the shapefile).

I turn both sides into the same string: lowercase, `&` → `and`, drop spaces and punctuation. Then `West Garo Hills` and `westgarohills` are the same key.

If the name ends with `district`, I strip that too, because some rows say `Pune District`.

In [4]:
def name_key(name):
    text = str(name or "").lower().strip()
    text = text.replace("&", "and")
    text = re.sub(r"[^a-z0-9]+", "", text)  # keep only letters and digits
    if text.endswith("district") and len(text) > len("district"):
        text = text[: -len("district")]
    return text


print(name_key("West Garo Hills"))
print(name_key("Pune District"))

westgarohills
pune


A few map labels are encoding junk. I only replace characters I actually saw in this GeoJSON.

In [5]:
def fix_map_junk(name):
    text = str(name or "")
    text = text.replace("#", "u")   # Bengal#ru -> Bengalu ru
    text = text.replace("@", "u")
    text = text.replace(">", "a")   # B>NKURA -> BANKURA
    text = text.replace("|", "i")
    text = text.replace("\\", "i")
    return text


def drop_brackets(name):
    # "Delhi (Rural)" -> "Delhi" so the extra word does not change the key
    return re.sub(r"\([^)]*\)", " ", str(name or ""))


print(name_key(fix_map_junk("Bengal#ru")))
print(name_key(fix_map_junk("B>NKURA")))

bengaluru
bankura


## 4. Spelling aliases

Census vs yearbook vs map: Ahmadabad / Ahmedabad, Mysore / Mysuru.
This is a short hand list, not a general spell-checker. Both directions so either side can be the map name.

In [6]:
aliases = {
    "ahmadabad": "ahmedabad",
    "ahmednagar": "ahilyanagar",
    "anugul": "angul",
    "badgam": "budgam",
    "bandipura": "bandipora",
    "baramula": "baramulla",
    "charkidadri": "charkhidadri",
    "chittaurgarh": "chittorgarh",
    "darang": "darrang",
    "dhaulpur": "dholpur",
    "eastsinghbum": "eastsinghbhum",
    "eastsinghbhum": "eastsinghbum",
    "jhunjhunun": "jhunjhunu",
    "kawardha": "kabirdham",
    "kabirdham": "kawardha",
    "keonjharkendujhar": "kendujhar",
    "mysuru": "mysore",
    "mysore": "mysuru",
    "narsinghpur": "narsimhapur",
    "siaha": "saiha",
    "saiha": "siaha",
    "thoothukkudi": "thoothukudi",
    "westgrohills": "westgarohills",
}

## 5. Last resort: unique close match

Levenshtein = number of single-letter edits to turn one string into another.
I only accept a fuzzy hit if **exactly one** map district is that close. If two names are equally close, I leave it unmatched rather than guess.

In [7]:
def edits(a, b):
    # standard Levenshtein: insert / delete / replace one character
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            insert = prev[j] + 1
            delete = cur[j - 1] + 1
            replace = prev[j - 1] + (0 if ca == cb else 1)
            cur.append(min(insert, delete, replace))
        prev = cur
    return prev[-1]


print(edits("ahmedabad", "ahmadabad"))  # 1 letter different

1


## 6. Read the map polygons

Each feature has a `district` name. I store:
- `district_key` — the join key
- `district` — a readable label (fix the `#` junk, title-case if the file is ALL CAPS)
- `state` from the same polygon

If two polygons collapse to the same key, I keep the first one.

In [8]:
with open(geo_path, encoding="utf-8") as f:
    geo = json.load(f)

geo_by_key = {}
for feat in geo.get("features") or []:
    props = feat.get("properties") or {}
    raw = props.get("district") or props.get("DISTRICT") or ""
    key = name_key(raw)
    if not key:
        continue
    if key in geo_by_key:
        continue  # first polygon wins

    nice = " ".join(fix_map_junk(raw).split())
    if nice.isupper() or "#" in str(raw) or ">" in str(raw):
        nice = nice.title()

    state = str(props.get("state") or props.get("STATE") or "").strip()
    if state.isupper():
        state = state.title()

    geo_by_key[key] = {"district": nice, "state": state, "geo_name_raw": raw}

geo_keys = set(geo_by_key)
print(len(geo_keys), "map districts with a usable name")

788 map districts with a usable name


## 7. Match one crop name to one map key

Order matters. I stop at the first unique hit:

1. exact key (raw name, GIS-fixed, brackets stripped)
2. alias list
3. unique prefix (`bankura` vs `bankurawest` — only if one polygon matches)
4. unique Levenshtein ≤ 2, and only if the lengths are close

Short names skip fuzzy matching so `Una` does not snap onto something random.

In [9]:
def keys_to_try(crop_name):
    seen = []
    for raw in (
        crop_name,
        fix_map_junk(crop_name),
        drop_brackets(crop_name),
        fix_map_junk(drop_brackets(crop_name)),
    ):
        key = name_key(raw)
        if key and key not in seen:
            seen.append(key)
    return seen


def match_to_map(crop_name):
    options = keys_to_try(crop_name)

    for key in options:
        if key in geo_keys:
            return key
        alias = aliases.get(key)
        if alias and alias in geo_keys:
            return alias
        # alias written the other way around
        for crop_spell, map_spell in aliases.items():
            if map_spell == key and crop_spell in geo_keys:
                return crop_spell
            if crop_spell == key and map_spell in geo_keys:
                return map_spell

    for key in options:
        if len(key) < 6:
            continue
        hits = [g for g in geo_keys if g.startswith(key) or key.startswith(g)]
        if len(hits) == 1:
            return hits[0]

    for key in options:
        if len(key) < 5:
            continue
        hits = [
            g for g in geo_keys
            if abs(len(g) - len(key)) <= 2 and edits(g, key) <= 2
        ]
        if len(hits) == 1:
            return hits[0]

    return None

In [10]:
# match each unique crop district once, then map it onto every row
mapping = {}
for name in df["district_name"].drop_duplicates():
    mapping[name] = match_to_map(name)

n_ok = sum(v is not None for v in mapping.values())
print("matched", n_ok, "of", len(mapping), "unique crop district names")

still_open = [n for n, k in mapping.items() if k is None]
print("unmatched examples:", still_open[:20])

matched 716 of 737 unique crop district names
unmatched examples: ['Sribhumi', 'Kolar', 'Belagavi', 'Bidar', 'Jajpur', 'Nuapada', 'Sonepur', 'Cooch Behar', 'Malda', 'Purulia', 'Purba Medinipur', 'Kabeerdham', 'Kangra', 'Anjaw', 'Reasi', 'Chikkaballapura', 'Balodabazar-Bhatapara', 'Balrampur-Ramanujganj', 'Alipurduar', 'Kumuram Bheem Asifabad']


## 8. Keep only rows that sit on a polygon

Unmatched names are saved so I can show the interviewer what I refused to guess.

In [11]:
df["district_key"] = df["district_name"].map(mapping)

unmatched = (
    df[df["district_key"].isna()]
    .groupby(["state_name", "district_name"], as_index=False)
    .size()
    .sort_values("size", ascending=False)
)
unmatched.to_csv(unmatched_path, index=False)
print("unmatched rows:", int(df["district_key"].isna().sum()))
unmatched.head(15)

unmatched rows: 15207


,state_name,district_name,size
7,Karnataka,Belagavi,1700
8,Karnataka,Bidar,1379
10,Karnataka,Kolar,1206
20,West Bengal,Purulia,1054
17,West Bengal,Cooch Behar,971
9,Karnataka,Chikkaballapura,933
18,West Bengal,Malda,860
4,Chhattisgarh,Kabeerdham,795
1,Assam,Sribhumi,776
12,Odisha,Nuapada,762


In [12]:
df = df[df["district_key"].notna()].copy()

# use the map's spelling so hover labels match the polygons
df["district"] = df["district_key"].map(lambda k: geo_by_key[k]["district"])
df["state"] = df["district_key"].map(lambda k: geo_by_key[k]["state"] or "")
df.loc[df["state"] == "", "state"] = df["state_name"]

for col in ["area", "production", "yield"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

## 9. Collapse duplicates and write

Same district + year + season + crop can appear more than once.
Area and production I **sum**. Yield I **average** — I do not recompute yield as production/area here, because the yearbook already published a yield figure.

In [13]:
out = (
    df.groupby(
        ["district_key", "district", "state", "year", "season", "crop_name"],
        as_index=False,
    )
    .agg(
        area_ha=("area", "sum"),
        production_tonnes=("production", "sum"),
        yield_t_ha=("yield", "mean"),
    )
    .rename(columns={"crop_name": "crop"})
    .sort_values(["district", "year", "season", "crop"])
)

out.to_csv(out_path, index=False)
print("wrote", len(out), "rows ->", out_path)
out.head()

wrote 424083 rows -> data\processed\crop_clean.csv


,district_key,district,state,year,season,crop,area_ha,production_tonnes,yield_t_ha
0,adilabad,Adilabad,Telangana,1997-1998,Kharif,Arhar/Tur,32200.0,1100.0,0.034
1,adilabad,Adilabad,Telangana,1997-1998,Kharif,Castor Seed,2600.0,700.0,0.269
2,adilabad,Adilabad,Telangana,1997-1998,Kharif,Cotton(Lint),144900.0,66500.0,0.459
3,adilabad,Adilabad,Telangana,1997-1998,Kharif,Dry Chillies,5500.0,3100.0,0.564
4,adilabad,Adilabad,Telangana,1997-1998,Kharif,Jowar,58200.0,31500.0,0.541
